In [0]:
from pyspark.sql import functions as F

BRONZE = "capstone_project_dev.bronze.raw_metadata"
SILVER = "capstone_project_dev.silver.validated_metadata"

print("Setup done")

In [0]:
df = spark.table(BRONZE)

print(f"Rows loaded : {df.count():,}")
print(f"Cols loaded : {len(df.columns)}")
df.show(3, truncate=30)

In [0]:
from pyspark.sql.window import Window

# Add timestamp for tiebreaking
df = df.withColumn("_processed_at", F.current_timestamp())

# Count nulls per row — fewer nulls = more complete row
required_fields = [
    "column_desc", "term_name", "data_steward",
    "security_classification", "certification_level"
]

null_score = sum(
    F.when(F.col(c).isNull(), 1).otherwise(0)
    for c in required_fields
)

df = df.withColumn("_null_score", null_score)

window = Window.partitionBy("table_name", "column_name") \
               .orderBy(F.col("_null_score").asc(), F.col("column_id").asc())

df = df.withColumn("_row_rank", F.row_number().over(window)) \
       .filter(F.col("_row_rank") == 1) \
       .drop("_row_rank", "_null_score")

print(f"Rows after dedup : {df.count():,}")

In [0]:
# Fix "Conf" → "Confidential" (dirty value found in bronze profiling)
df = df.withColumn("security_classification",
    F.when(
        F.trim(F.col("security_classification")) == "Conf",
        F.lit("Confidential")
    ).otherwise(F.col("security_classification"))
)

# Make sure pii_flag and critical_data_element_flag are proper booleans
df = df.withColumn("pii_flag",
    F.col("pii_flag").cast("boolean")
).withColumn("critical_data_element_flag",
    F.col("critical_data_element_flag").cast("boolean")
)

# Make sure total_record_count and invalid_record_count are proper numbers
df = df.withColumn("total_record_count",
    F.col("total_record_count").cast("long")
).withColumn("invalid_record_count",
    F.col("invalid_record_count").cast("long")
)

print("Data cleaned")

In [0]:
# Count distinct columns per table
cols_per_table = df.groupBy("table_name").agg(
    F.countDistinct("column_name").alias("column_count")
)

# Calculate threshold dynamically — mean minus 1 standard deviation
stats = cols_per_table.agg(
    F.mean("column_count").alias("mean"),
    F.stddev("column_count").alias("stddev")
).collect()[0]

threshold = stats["mean"] - stats["stddev"]

print(f"Mean columns per table  : {stats['mean']:.1f}")
print(f"Stddev                  : {stats['stddev']:.1f}")
print(f"Threshold (mean-1std)   : {threshold:.1f}")

# Join back so every row knows its table's column count and threshold
df = df.join(cols_per_table, on="table_name", how="left")
df = df.withColumn("col_standard_threshold", F.lit(threshold))

print("Column count per table joined")
cols_per_table.orderBy("column_count").show()

In [0]:
df = df.withColumn("rule01_mandatory_null",
    F.col("column_id").isNull() |
    F.col("column_name").isNull() |
    F.col("table_id").isNull() |
    F.col("table_name").isNull()
)

df = df.withColumn("rule02_pii_flag_invalid",
    F.col("pii_flag").isNull()
)

df = df.withColumn("rule03_security_invalid",
    F.col("security_classification").isNull() |
    ~F.col("security_classification").isin("Internal", "Confidential", "Public")
)

df = df.withColumn("rule04_cert_level_invalid",
    F.col("certification_level").isNull() |
    ~F.col("certification_level").isin("Registered", "Certified", "Documented")
)
df = df.withColumn("rule05_pii_not_confidential",
    (F.col("pii_flag") == True) &
    (F.col("security_classification").isNull() |
    (F.col("security_classification") != "Confidential"))
)

df = df.withColumn("rule06_below_col_standard",
    F.col("column_count") < F.col("col_standard_threshold")
)

df = df.withColumn("rule07_cde_no_steward",
    (F.col("critical_data_element_flag") == True) &
    F.col("data_steward").isNull()
)

df = df.withColumn("rule08_invalid_count_exceeds_total",
    F.col("invalid_record_count") > F.col("total_record_count")
)

print("All rules applied")

In [0]:
df = df.withColumn("compliance_flag",
    F.when(
        F.col("rule01_mandatory_null")             |
        F.col("rule02_pii_flag_invalid")           |
        F.col("rule03_security_invalid")           |
        F.col("rule04_cert_level_invalid")         |
        F.col("rule05_pii_not_confidential")       |
        F.col("rule06_below_col_standard")         |
        F.col("rule07_cde_no_steward")             |
        F.col("rule08_invalid_count_exceeds_total"),
        "NON-COMPLIANT"
    ).otherwise("COMPLIANT")
)

print("Compliance flag added")

In [0]:
total = df.count()
rules = [
    "rule01_mandatory_null",
    "rule02_pii_flag_invalid",
    "rule03_security_invalid",
    "rule04_cert_level_invalid",
    "rule05_pii_not_confidential",
    "rule06_below_col_standard",
    "rule07_cde_no_steward",
    "rule08_invalid_count_exceeds_total",
]

print(f"{'Rule':<45} {'Violations':>12}    {'Status'}")
print("─" * 70)
for rule in rules:
    count = df.filter(F.col(rule) == True).count()
    status = "PASS" if count == 0 else "FAIL"
    print(f"{rule:<45} {count:>12,}    {status}")

print("─" * 70)
compliant     = df.filter(F.col("compliance_flag") == "COMPLIANT").count()
non_compliant = df.filter(F.col("compliance_flag") == "NON-COMPLIANT").count()
print(f"\nCOMPLIANT rows     : {compliant:,}")
print(f"NON-COMPLIANT rows : {non_compliant:,}")
print(f"Total rows         : {total:,}")

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(SILVER)

print(f"Silver table written: {SILVER}")

In [0]:
verify = spark.table(SILVER)
print(f"Silver row count : {verify.count():,}")
print(f"Silver col count : {len(verify.columns)}")

verify.select(
    "table_name", "column_name", "pii_flag",
    "security_classification", "compliance_flag",
    "rule05_pii_not_confidential",
    "rule07_cde_no_steward"
).show(10, truncate=30)